# Online Retails Purchase

### Introduction:



### Step 1. Import the necessary libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

%matplotlib inline


### Step 2. Import the dataset from this [address](https://raw.githubusercontent.com/thieu1995/csv-files/main/data/pandas/Online_Retail.csv).

### Step 3. Assign it to a variable called online_rt
Note: if you receive a utf-8 decode error, set `encoding = 'latin1'` in `pd.read_csv()`.

In [ ]:
online_rt = pd.read_csv("online_retail.csv", encoding="latin1")
online_rt.head()


### Step 4. Create a histogram with the 10 countries that have the most 'Quantity' ordered except UK

In [ ]:
country_quantity = (
    online_rt[online_rt["Country"] != "United Kingdom"]
    .groupby("Country")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

ax = country_quantity.plot(kind="bar", figsize=(10, 4), color="steelblue")
ax.set_xlabel("Country")
ax.set_ylabel("Quantity ordered")
ax.set_title("Top 10 Countries by Quantity Ordered")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Step 5.  Exclude negative Quantity entries

In [ ]:
online_rt = online_rt[online_rt["Quantity"] > 0].copy()
online_rt.head()


### Step 6. Create a scatterplot with the Quantity per UnitPrice by CustomerID for the top 3 Countries (except UK)

In [ ]:
top3_countries = (
    online_rt[online_rt["Country"] != "United Kingdom"]
    .groupby("Country")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
    .index
)

fig, ax = plt.subplots(figsize=(8, 5))
for country in top3_countries:
    subset = online_rt[online_rt["Country"] == country]
    ax.scatter(subset["UnitPrice"], subset["Quantity"], alpha=0.25, label=country)

ax.set_xlabel("UnitPrice")
ax.set_ylabel("Quantity")
ax.set_title("Quantity per UnitPrice for Top 3 Countries")
ax.legend()
plt.show()


### Step 7. Investigate why the previous results look so uninformative.

This section might seem a bit tedious to go through. But I've thought of it as some kind of a simulation of problems one might encounter when dealing with data and other people. Besides there is a prize at the end (i.e. Section 8).

(But feel free to jump right ahead into Section 8 if you want; it doesn't require that you finish this section.)

#### Step 7.1 Look at the first line of code in Step 6. And try to figure out if it leads to any kind of problem.
##### Step 7.1.1 Display the first few rows of that DataFrame.

In [ ]:
online_rt[online_rt["Country"].isin(top3_countries)].head()


##### Step 7.1.2 Think about what that piece of code does and display the dtype of `UnitPrice`

In [ ]:
online_rt["UnitPrice"].dtype


##### Step 7.1.3 Pull data from `online_rt`for `CustomerID`s 12346.0 and 12347.0.

In [ ]:
online_rt[online_rt["CustomerID"].isin([12346.0, 12347.0])].sort_values(
    ["CustomerID", "InvoiceDate"]
)


#### Step 7.2 Reinterpreting the initial problem.

To reiterate the question that we were dealing with:  
"Create a scatterplot with the Quantity per UnitPrice by CustomerID for the top 3 Countries"

The question is open to a set of different interpretations.
We need to disambiguate.

We could do a single plot by looking at all the data from the top 3 countries.
Or we could do one plot per country. To keep things consistent with the rest of the exercise,
let's stick to the latter oprion. So that's settled.

But "top 3 countries" with respect to what? Two answers suggest themselves:
Total sales volume (i.e. total quantity sold) or total sales (i.e. revenue).
This exercise goes for sales volume, so let's stick to that.

##### Step 7.2.1 Find out the top 3 countries in terms of sales volume.

In [ ]:
top3_countries = (
    online_rt[online_rt["Country"] != "United Kingdom"]
    .groupby("Country")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
    .index
    .tolist()
)
top3_countries


##### Step 7.2.2 

Now that we have the top 3 countries, we can focus on the rest of the problem:  
"Quantity per UnitPrice by CustomerID".  
We need to unpack that.

"by CustomerID" part is easy. That means we're going to be plotting one dot per CustomerID's on our plot. In other words, we're going to be grouping by CustomerID.

"Quantity per UnitPrice" is trickier. Here's what we know:  
*One axis will represent a Quantity assigned to a given customer. This is easy; we can just plot the total  Quantity for each customer.  
*The other axis will represent a UnitPrice assigned to a given customer. Remember a single customer can have any number of orders with different prices, so summing up prices isn't quite helpful. Besides it's not quite clear what we mean when we say "unit price per customer"; it sounds like price of the customer! A reasonable alternative is that we assign each customer the average amount each has paid per item. So let's settle that question in that manner.

#### Step 7.3 Modify, select and plot data
##### Step 7.3.1 Add a column to online_rt called `Revenue` calculate the revenue (Quantity * UnitPrice) from each sale.
We will use this later to figure out an average price per customer.

In [ ]:
online_rt["Revenue"] = online_rt["Quantity"] * online_rt["UnitPrice"]
online_rt.head()


##### Step 7.3.2 Group by `CustomerID` and `Country` and find out the average price (`AvgPrice`) each customer spends per unit.

In [ ]:
customer_country = (
    online_rt.dropna(subset=["CustomerID"])
    .groupby(["CustomerID", "Country"])
    .agg(Quantity=("Quantity", "sum"), Revenue=("Revenue", "sum"))
    .reset_index()
)
customer_country["AvgPrice"] = customer_country["Revenue"] / customer_country["Quantity"]
customer_country.head()


##### Step 7.3.3 Plot

In [ ]:
fig, axes = plt.subplots(1, len(top3_countries), figsize=(15, 4), sharey=True)

for ax, country in zip(axes, top3_countries):
    subset = customer_country[customer_country["Country"] == country]
    ax.scatter(subset["AvgPrice"], subset["Quantity"], alpha=0.45)
    ax.set_title(country)
    ax.set_xlabel("Average price")
    ax.set_ylabel("Quantity")

plt.tight_layout()
plt.show()


#### Step 7.4 What to do now?
We aren't much better-off than what we started with. The data are still extremely scattered around and don't seem quite informative.

But we shouldn't despair!
There are two things to realize:
1) The data seem to be skewed towaards the axes (e.g. we don't have any values where Quantity = 50000 and AvgPrice = 5). So that might suggest a trend.
2) We have more data! We've only been looking at the data from 3 different countries and they are plotted on different graphs.

So: we should plot the data regardless of `Country` and hopefully see a less scattered graph.

##### Step 7.4.1 Plot the data for each `CustomerID` on a single graph

In [ ]:
customer_totals = (
    online_rt.dropna(subset=["CustomerID"])
    .groupby("CustomerID")
    .agg(Quantity=("Quantity", "sum"), Revenue=("Revenue", "sum"))
    .reset_index()
)
customer_totals["AvgPrice"] = customer_totals["Revenue"] / customer_totals["Quantity"]

ax = customer_totals.plot.scatter(
    x="AvgPrice", y="Quantity", figsize=(8, 5), alpha=0.35
)
ax.set_title("Quantity vs Average Price by Customer")
plt.show()


##### Step 7.4.2 Zoom in so we can see that curve more clearly

In [ ]:
zoomed = customer_totals[
    (customer_totals["AvgPrice"] <= 20) & (customer_totals["Quantity"] <= 5000)
]

ax = zoomed.plot.scatter(x="AvgPrice", y="Quantity", figsize=(8, 5), alpha=0.35)
ax.set_title("Quantity vs Average Price by Customer, Zoomed")
plt.show()


### 8. Plot a line chart showing revenue (y) per UnitPrice (x).

Did Step 7 give us any insights about the data? Sure! As average price increases, the quantity ordered decreses.  But that's hardly surprising. It would be surprising if that wasn't the case!

Nevertheless the rate of drop in quantity is so drastic, it makes me wonder how our revenue changes with respect to item price. It would not be that surprising if it didn't change that much. But it would be interesting to know whether most of our revenue comes from expensive or inexpensive items, and how that relation looks like.

That is what we are going to do now.

#### 8.1 Group `UnitPrice` by intervals of 1 for prices [0,50), and sum `Quantity` and `Revenue`.

In [ ]:
price_bins = pd.cut(online_rt["UnitPrice"], bins=range(0, 51), right=False)
price_groups = (
    online_rt.assign(PriceRange=price_bins)
    .dropna(subset=["PriceRange"])
    .groupby("PriceRange", observed=True)
    .agg(Quantity=("Quantity", "sum"), Revenue=("Revenue", "sum"))
    .reset_index()
)
price_groups["UnitPrice"] = price_groups["PriceRange"].apply(lambda interval: interval.left)
price_groups.head()


#### 8.3 Plot.

In [ ]:
ax = price_groups.plot(x="UnitPrice", y="Revenue", kind="line", figsize=(10, 4))
ax.set_xlabel("UnitPrice")
ax.set_ylabel("Revenue")
ax.set_title("Revenue by UnitPrice Interval")
plt.show()


#### 8.4 Make it look nicer.
x-axis needs values.  
y-axis isn't that easy to read; show in terms of millions.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(price_groups["UnitPrice"], price_groups["Revenue"], marker="o")
ax.set_xlabel("UnitPrice")
ax.set_ylabel("Revenue")
ax.set_title("Revenue by UnitPrice Interval")
ax.set_xticks(range(0, 51, 5))
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, pos: f"{value / 1_000_000:.1f}M"))
ax.grid(True, alpha=0.25)
plt.show()


### BONUS: Create your own question and answer it.

In [ ]:
# Bonus: Which countries generate the most revenue outside the UK?
country_revenue = (
    online_rt[online_rt["Country"] != "United Kingdom"]
    .groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

ax = country_revenue.plot(kind="bar", figsize=(10, 4), color="seagreen")
ax.set_xlabel("Country")
ax.set_ylabel("Revenue")
ax.set_title("Top Countries by Revenue, excluding UK")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
